# importing

In [ ]:
import pandas as pd
import numpy as np
import os
import sys
import librosa
import librosa.display
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import train_test_split

import IPython.display as ipd
from IPython.display import Audio

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (Dense, Embedding, LSTM, BatchNormalization,GRU, Input, Flatten, Dropout, Activation, Conv1D, MaxPooling1D, AveragePooling1D)
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.callbacks import ModelCheckpoint

from audio_pipeline import extract_features_from_audio_array
from audio_pipeline import extract_features_from_file
from config import TARGET_SR, MONO

import warnings
if not sys.warnoptions:
    warnings.simplefilter("ignore")
warnings.filterwarnings("ignore", category=DeprecationWarning) 

print("Done - TensorFlow version:", tf.__version__)

# Integration with Azure

In [ ]:
!pip install azure-storage-blob
!pip install seaborn python-dotenv librosa scikit-learn

In [ ]:
from azure.storage.blob import BlobServiceClient
import pandas as pd
import os
from dotenv import load_dotenv

load_dotenv()

connection_string = os.getenv("AZURE_STORAGE_CONNECTION_STRING")
container_name = "wav-files"

blob_service_client = BlobServiceClient.from_connection_string(connection_string)
container_client = blob_service_client.get_container_client(container_name)


def get_df_from_blob_dataset(prefix):
    emotions = []
    file_paths = []

    blobs = container_client.list_blobs(name_starts_with=prefix)

    for blob in blobs:
        if blob.name.endswith(".wav"):
            
            filename = blob.name.split("/")[-1]
            parts = filename.split('-')

            if len(parts) >= 3:
                emotion = parts[2].lower()
                emotions.append(emotion)
                file_paths.append(blob.name)

    return pd.DataFrame({
        'Emotions': emotions,
        'Path': file_paths
    })


ravdess_df = get_df_from_blob_dataset("RAVDESS/")
crema_df   = get_df_from_blob_dataset("CREMAD/")
tess_df    = get_df_from_blob_dataset("TESS/")
savee_df   = get_df_from_blob_dataset("SAVEE/")

data_path = pd.concat(
    [ravdess_df, crema_df, tess_df, savee_df],
    axis=0
)

data_path.reset_index(drop=True, inplace=True)

data_path.to_csv("data_path.csv", index=False)

print(data_path.head())
print("-" * 30)
print(data_path.Emotions.value_counts())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.title('Count of Emotions', size=16)
sns.countplot(data_path.Emotions)
plt.ylabel('Emotions', size=12)
plt.xlabel('Count', size=12)
sns.despine(top=True, right=True, left=False, bottom=False)
plt.show()

# Data Augmentation

In [ ]:
import os
import pandas as pd
import numpy as np
import librosa
import warnings
import time
import io
from azure.storage.blob import BlobServiceClient

warnings.filterwarnings('ignore')

df = pd.read_csv("data_path.csv")

CONNECTION_STRING = os.getenv("AZURE_STORAGE_CONNECTION_STRING")
CONTAINER_NAME = "wav-files"

if not CONNECTION_STRING:
    raise ValueError("Connection String not found. Make sure you defined the AZURE_STORAGE_CONNECTION_STRING environment variable.")

blob_service_client = BlobServiceClient.from_connection_string(CONNECTION_STRING)

# Data Augmentation
def noise(data):
    noise_amp = 0.035 * np.random.uniform() * np.amax(data)
    return data + noise_amp * np.random.normal(size=data.shape[0])

def stretch(data, rate=0.8):
    return librosa.effects.time_stretch(y=data, rate=rate)

def shift(data):
    shift_range = int(np.random.uniform(low=-5, high=5) * 1000)
    return np.roll(data, shift_range)

def pitch(data, sampling_rate, pitch_factor=0.7):
    return librosa.effects.pitch_shift(y=data, sr=sampling_rate, n_steps=pitch_factor)


# Feature Extraction of 30 features using MFCC algo
def extract_features(data, sr):
    return extract_features_from_audio_array(data, sr)


def get_features(path):
    try:
        clean_path = path.replace('\\', '/')

        blob_client = blob_service_client.get_blob_client(
            container=CONTAINER_NAME,
            blob=clean_path
        )

        download_stream = blob_client.download_blob()
        audio_file = io.BytesIO(download_stream.readall())

        data, sr = librosa.load(audio_file, sr=TARGET_SR, mono=MONO)
        data = data.astype(np.float32)

        feature_list = [
            extract_features_from_audio_array(data, sr),
            extract_features_from_audio_array(noise(data), sr),
            extract_features_from_audio_array(stretch(data), sr),
            extract_features_from_audio_array(shift(data), sr),
            extract_features_from_audio_array(pitch(data, sr), sr),
        ]

        return feature_list

    except Exception as e:
        print(f"Error processing {path}: {e}")
        return None


X, Y = [], []  # X- list of vector for each file , y- real emotion of each file 

print(f"Starting to process {len(df)} files from Azure (Secure).")
start_time = time.time()

for index, row in df.iterrows():
    path = row['Path']
    emotion = row['Emotions']
    
    features = get_features(path)
    
    if features is not None:
        for ele in features:
            X.append(ele)
            Y.append(emotion)
            
    if (index + 1) % 100 == 0:
        print(f"Processed {index + 1}/{len(df)} files from the cloud...")

end_time = time.time()
print(f"Finished! Execution time: {round((end_time - start_time)/60, 2)} minutes.")


Features_df = pd.DataFrame(X)
Features_df['Labels'] = Y

Features_df.to_csv("features_ready_for_model.csv", index=False)
print(f"Data saved successfully. Total samples for training: {len(Features_df)}")

# Data Preperation

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
import pickle

Emotions = pd.read_csv('features_ready_for_model.csv')
# Handle missing values (NaN) by replacing them with 0
Emotions = Emotions.fillna(0)

#Separate Features (X) and Labels (Y)
X = Emotions.iloc[:, :-1].values
Y = Emotions['Labels'].values

print(f"Original data shapes - X: {X.shape}, Y: {Y.shape}")

# One-Hot Encoding for the labels (Y)
encoder = OneHotEncoder() #technice thay encodin emotion to vector of 0,1 - we dont encode emotions by 1,2,3 bcs the model may think 3 is bigger than 1 and we want treat all categories equally
Y = encoder.fit_transform(np.array(Y).reshape(-1, 1)).toarray() 

#Train/Test Split (80% training, 20% testing)
x_train, x_test, y_train, y_test = train_test_split(X, Y, random_state=42, test_size=0.2, shuffle=True) # 20% of the data will be used for testing , shuffle - avoid overfitting
print(f"Train set: X={x_train.shape}, Y={y_train.shape}")
print(f"Test set: X={x_test.shape}, Y={y_test.shape}")

# Feature Scaling (Standardization)
print("Scaling features...")
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test) # Only transform for test set to avoid data leakage!

# Reshape data for 1D CNN (Adding a 3rd dimension)
print("Reshaping data for CNN...")
x_traincnn = np.expand_dims(x_train, axis=2)#axis =2 because we want to add a third dimension to the data (it's not stereo) - audio + mfcc vector
x_testcnn = np.expand_dims(x_test, axis=2)

print(f"Final shapes for CNN - Train: {x_traincnn.shape}, Test: {x_testcnn.shape}")

# Save Scaler and Encoder for future 
print("Saving scaler and encoder objects...")
with open('scaler.pickle', 'wb') as f:
    pickle.dump(scaler, f)
with open('encoder.pickle', 'wb') as f:
    pickle.dump(encoder, f)

print("Data is ready for the model.")

# CNN Arcticeture

In [ ]:
import tensorflow as tf
import tensorflow.keras.layers as L

print("Building the 1D CNN Architecture...")

model = tf.keras.Sequential([
    L.Conv1D(512, kernel_size=5, strides=1, padding='same', activation='relu', input_shape=(x_traincnn.shape[1], 1)),
    L.BatchNormalization(),
    L.MaxPool1D(pool_size=5, strides=2, padding='same'),
    
    L.Conv1D(512, kernel_size=5, strides=1, padding='same', activation='relu'),
    L.BatchNormalization(),
    L.MaxPool1D(pool_size=5, strides=2, padding='same'),
    L.Dropout(0.2), 
    
    L.Conv1D(256, kernel_size=5, strides=1, padding='same', activation='relu'),
    L.BatchNormalization(),
    L.MaxPool1D(pool_size=5, strides=2, padding='same'),
    
    L.Conv1D(256, kernel_size=3, strides=1, padding='same', activation='relu'),
    L.BatchNormalization(),
    L.MaxPool1D(pool_size=5, strides=2, padding='same'),
    L.Dropout(0.2),  
    
    L.Conv1D(128, kernel_size=3, strides=1, padding='same', activation='relu'),
    L.BatchNormalization(),
    L.MaxPool1D(pool_size=3, strides=2, padding='same'),
    L.Dropout(0.2),  
    
    L.Flatten(),
    L.Dense(512, activation='relu'),
    L.BatchNormalization(),
    L.Dropout(0.2),
    
    L.Dense(4, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

# Training Setup

In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau



model_checkpoint = ModelCheckpoint('best_cnn_weights.h5', monitor='val_accuracy', save_best_only=True, verbose=1) #checkpoint- save the best model and delete the rest

#Early Stopping: Stops training if the model doesn't improve for 10 epochs
early_stop = EarlyStopping(monitor='val_accuracy', mode='max', patience=10, restore_best_weights=True, verbose=1)

lr_reduction = ReduceLROnPlateau(monitor='val_accuracy', patience=5, verbose=1, factor=0.5, min_lr=0.00001) #if the model doesn't improve for 5 epochs, reduce the steps by 50% (gradient descent)

print("Callbacks are ready!")


# CNN Model

In [ ]:
import matplotlib.pyplot as plt


# --- The actual training step ---
history = model.fit(
    x_traincnn, y_train, 
    epochs=50, #epoch - one pass through the entire dataset (~30K samples) 
    validation_data=(x_testcnn, y_test), 
    batch_size=64, #take 64 samples each time
    callbacks=[early_stop, lr_reduction, model_checkpoint]
)

print(" Model has finished training.")


print("Plotting training history...")
epochs = range(len(history.history['accuracy']))
fig, ax = plt.subplots(1, 2, figsize=(20, 6))

# Plotting Loss
ax[0].plot(epochs, history.history['loss'], label='Training Loss', color='blue')
ax[0].plot(epochs, history.history['val_loss'], label='Validation Loss', color='red')
ax[0].set_title('Training & Validation Loss')
ax[0].legend()
ax[0].set_xlabel("Epochs")

# Plotting Accuracy
ax[1].plot(epochs, history.history['accuracy'], label='Training Accuracy', color='blue')
ax[1].plot(epochs, history.history['val_accuracy'], label='Validation Accuracy', color='red')
ax[1].set_title('Training & Validation Accuracy')
ax[1].legend()
ax[1].set_xlabel("Epochs")

plt.show()

In [ ]:
# predicting on test data.
pred_test0 = model.predict(x_testcnn)
y_pred0 = encoder.inverse_transform(pred_test0)
y_test0 = encoder.inverse_transform(y_test)

# Check for random predictions
df0 = pd.DataFrame(columns=['Predicted Labels', 'Actual Labels'])
df0['Predicted Labels'] = y_pred0.flatten()
df0['Actual Labels'] = y_test0.flatten()

df0.head(10)

# Evaluation

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt


pred_test = model.predict(x_testcnn)

y_pred_labels = encoder.inverse_transform(pred_test)
y_test_labels = encoder.inverse_transform(y_test)

cm = confusion_matrix(y_test_labels, y_pred_labels)

labels = encoder.categories_[0]

plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples', 
            xticklabels=labels, yticklabels=labels)

plt.title('Confusion Matrix - Audio Emotion Recognition', size=20)
plt.xlabel('Predicted Labels (What the model thought)', size=14)
plt.ylabel('Actual Labels (The Truth)', size=14)
plt.show()

print("\n--- Detailed Classification Report ---")
print(classification_report(y_test_labels, y_pred_labels))

# Saving the model

In [ ]:
import os
import numpy as np
import librosa
import pickle
from tensorflow.keras.models import load_model


FOLDER_NAME = 'model_assets'

if not os.path.exists(FOLDER_NAME):
    os.makedirs(FOLDER_NAME)
    print(f"Directory created: {FOLDER_NAME}")

model_path = os.path.join(FOLDER_NAME, 'final_emotion_model.keras')
model.save(model_path)
print(f"Model saved to: {model_path}")


with open(os.path.join(FOLDER_NAME, 'scaler.pickle'), 'wb') as f:
    pickle.dump(scaler, f)

with open(os.path.join(FOLDER_NAME, 'encoder.pickle'), 'wb') as f:
    pickle.dump(encoder, f)

print(f"Preprocessing tools saved to: {FOLDER_NAME}/")


def extract_features_single(path):
    return extract_features_from_file(path)

def predict_emotion(audio_path):
    # Step A: Extract features
    feat = extract_features_single(audio_path)
    
    # Step B: Scale features using the scaler we saved earlier
    feat = scaler.transform(feat.reshape(1, -1))
    
    # Step C: Reshape for CNN (Add the 3rd dimension)
    feat = np.expand_dims(feat, axis=2)
    
    # Step D: Predict probabilities
    predictions = model.predict(feat, verbose=0)
    
    # Step E: Get the emotion label and confidence
    emotion = encoder.inverse_transform(predictions)
    confidence = np.max(predictions)
    
    return emotion[0][0], confidence

print("\nSystem ready for new predictions using assets from the folder!")